# Module 9: Dataset and evaluation harness

**Project:** collect and label a dataset for a task of your choice, write the **labeling guide**, and build an **evaluation harness**: one function that runs a model on a fixed test set and prints trustworthy metrics. Then find out *where* the model fails and test a fix.

**Done when:**
- you labeled **at least 200 examples** and wrote a **one-page labeling guide**,
- one call, `run_eval(...)`, runs a model on your **fixed test set** and prints accuracy, precision, recall and F1, **without any manual step**,
- you **categorized at least 10 failures**, proposed a fix for the largest category, and measured the fix **on the same test set**.

**How to use this notebook**
1. Save your own copy (*File > Save a copy in Drive*). A free CPU runtime is enough.
2. Run cells from top to bottom. Fill in every `YOUR CODE HERE`. The **Self-check** cells tell you if you got it right.
3. Stuck? Use the hint ladder: docs, then the hints below each step, then the course forum. Only after your own attempt, open the solution page.

**Why this module matters:** a model is only as good as the data it learns from and the test that judges it. Most failed AI products fail here, not in the model.

## Step 0: Setup (given)

In [ ]:
import json, csv, hashlib, re, time, random
from collections import Counter
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

random.seed(0); np.random.seed(0)
print("Ready.")

## Step 1: Choose a task and write the labeling guide

Pick a **classification task** that you can label yourself, ideally one with **two labels** where one label is the rare, important one. Examples: *urgent* versus *normal* support tickets, *spam* versus *not spam*, *defective* versus *fine* product notes, *complaint* versus *other* reviews.

Your **labeling guide** is the document that makes labels consistent. Two people (or you, next month) must reach the same label for the same item. A good guide has:
- a **definition** of each label,
- **examples** of each label,
- **edge cases**: borderline items and the rule for deciding them.

Write it in the variable `LABEL_GUIDE` (a text block). **Option B (demo):** set `USE_DEMO = True` for a ready-made task about support tickets. Write the guide for it anyway, because writing the guide is the exercise.

In [ ]:
USE_DEMO = False
CSV_PATH = Path("my_labeled_data.csv")    # columns: text, label  (optional: channel, label_b)
POSITIVE = "urgent"                       # the rare label you care most about finding. CHANGE it for your own task
SLICE_KEY = "channel"                     # an optional column used for the slice check. Set to None if you have none

LABEL_GUIDE = """
YOUR CODE HERE: write your labeling guide here (about one page).
"""

## Step 2: The dataset

**Your own data:** put it in `my_labeled_data.csv` with columns `text` and `label` (and optionally `channel`, a column you want to check for fairness, and `label_b`, the label from a second person). Aim for **at least 200 examples** and consider where the examples come from: they should look like the real inputs your product will see.

**Demo:** the generator below creates 1,200 support tickets, with a rare *urgent* class, two writing styles (email and chat), and a second labeler.

In [ ]:
def make_ticket_dataset(n=500, seed=0):
    """Synthetic support tickets. label: 'urgent' or 'normal' (about 12% urgent). Two labelers (label, label_b) and a channel (email or chat)."""
    rng = random.Random(seed)
    urgent = ["our whole site is down and customers cannot check out", "we cannot log in and our deadline is in one hour",
              "payment processing is failing for every order", "we lost all our data after the update, please help now",
              "I think someone hacked my account and changed my password", "the app crashes on startup for all our staff",
              "customers are being charged twice, this is critical", "server error 500 on every page since this morning"]
    normal = ["how do I change my billing address", "thanks for the quick reply yesterday", "can you add a dark mode in a future update",
              "what payment methods do you accept", "I would like to know how to export my report", "great product, just a small suggestion about the menu",
              "where can I find my invoice from last month", "do you offer a discount for students", "please update my email address on file",
              "how long does shipping usually take", "I forgot how to reset my password, no rush", "is there a mobile version of the dashboard",
              "login page looks different today, just curious why", "a payment reminder arrived, I wanted to confirm it is correct",
              "the site is a bit slow today but it works"]
    tricky_normal = ["this is not urgent but could you explain the invoice", "URGENT!!! just kidding, I only wanted to say thank you",
                     "no hurry at all, a question about the payment page"]
    borderline = ["login is slow for some of our team", "one customer saw a payment error once", "the export feature fails sometimes",
                  "the page crashed for me twice today"]
    openers = ["", "hi, ", "hello team, ", "good morning, ", "hey, "]
    closers = ["", " thanks", " please advise", " thank you", " regards"]
    out, seen = [], set()
    while len(out) < n:
        r = rng.random()
        if r < 0.10:
            text, lab = rng.choice(urgent), "urgent"; lab_b = lab
        elif r < 0.14:
            text = rng.choice(borderline); lab = "urgent" if rng.random() < 0.5 else "normal"; lab_b = "urgent" if rng.random() < 0.5 else "normal"
        elif r < 0.20:
            text, lab = rng.choice(tricky_normal), "normal"; lab_b = lab
        else:
            text, lab = rng.choice(normal), "normal"; lab_b = lab
        channel = "chat" if rng.random() < 0.4 else "email"
        text = rng.choice(openers) + text + rng.choice(closers)
        if channel == "email":
            text = text[0].upper() + text[1:] + "."
        else:
            text = text.lower().replace(",", "")
            words = text.split()
            if len(words) > 6 and rng.random() < 0.6: text = " ".join(words[:rng.randint(4, 6)])
        if text in seen: 
            text += " " + str(rng.randint(2, 99))
        seen.add(text)
        out.append({"text": text, "label": lab, "label_b": lab_b, "channel": channel})
    return out


def load_data():
    if USE_DEMO:
        return make_ticket_dataset(1200, seed=0)
    assert CSV_PATH.exists(), "my_labeled_data.csv not found. Upload it (columns: text, label) or set USE_DEMO = True."
    with open(CSV_PATH, newline="", encoding="utf-8") as f:
        return [dict(r) for r in csv.DictReader(f)]

examples = load_data()
print(len(examples), "examples. First three:")
for e in examples[:3]:
    print("  ", e)

### A data quality report

Before you train anything, look at your data. Write `data_quality_report(examples)`. It returns a dictionary with:
- `n`: the number of examples,
- `label_counts`: a dictionary label -> count,
- `imbalance_ratio`: the count of the most common label divided by the count of the least common one,
- `n_duplicates`: the number of examples whose text (after `strip()`) has **already appeared earlier** in the list,
- `n_empty`: the number of examples whose text is empty after `strip()`.

In [ ]:
def data_quality_report(examples):
    """Summarize the data: n, label_counts, imbalance_ratio, n_duplicates, n_empty."""
    # YOUR CODE HERE
    return {"n": 0, "label_counts": {}, "imbalance_ratio": 0.0, "n_duplicates": 0, "n_empty": 0}

### Self-check: data quality report

In [ ]:
_ex = [{"text": "a", "label": "x"}, {"text": "a ", "label": "x"}, {"text": "  ", "label": "y"}, {"text": "bb", "label": "x"}, {"text": "c", "label": "x"}]
_r = data_quality_report(_ex)
assert _r["n"] == 5, "n should be the number of examples."
assert _r["label_counts"] == {"x": 4, "y": 1}, f"label_counts should be {{'x': 4, 'y': 1}} but is {_r['label_counts']}."
assert abs(_r["imbalance_ratio"] - 4.0) < 1e-9, f"imbalance_ratio should be 4.0 (4 x versus 1 y) but is {_r['imbalance_ratio']}."
assert _r["n_duplicates"] == 1, f"n_duplicates should be 1 ('a' and 'a ' are the same text after strip()) but is {_r['n_duplicates']}."
assert _r["n_empty"] == 1, f"n_empty should be 1 (the whitespace-only text) but is {_r['n_empty']}."
print("data_quality_report() check passed.")

Now look at **your** data. This cell is given. It also removes empty texts and exact duplicates, because a duplicate that lands in both the training and test sets would inflate your score.

In [ ]:
report0 = data_quality_report(examples)
print("Before cleaning:", report0)

seen, cleaned = set(), []
for e in examples:
    t = e["text"].strip()
    if t and t not in seen:
        seen.add(t); cleaned.append({**e, "text": t})
examples = cleaned
report1 = data_quality_report(examples)
print("After cleaning: ", report1)
print(f"\nLabel balance: {report1['label_counts']}  (the rare label is {100 * report1['label_counts'][POSITIVE] / report1['n']:.0f}% of the data)")

### Self-check: dataset and labeling guide

In [ ]:
assert report1["n"] >= 200, f"You have {report1['n']} examples after cleaning. You need at least 200."
assert len(report1["label_counts"]) >= 2, "You need at least two different labels."
assert POSITIVE in report1["label_counts"], f"POSITIVE = {POSITIVE!r} is not one of your labels {list(report1['label_counts'])}."
assert report1["n_duplicates"] == 0 and report1["n_empty"] == 0, "Duplicates or empty texts are still in the data."
_g = LABEL_GUIDE.lower()
assert "your code here" not in _g and len(LABEL_GUIDE.strip()) >= 400, "Write a real labeling guide of about one page (at least 400 characters)."
for _lab in report1["label_counts"]:
    assert _lab.lower() in _g, f"Your labeling guide never mentions the label {_lab!r}. Define every label."
assert "example" in _g, "Include examples of each label in your guide."
assert any(w in _g for w in ("edge", "borderline", "unclear", "ambiguous", "tricky")), "Describe the edge cases (borderline items) and the rule for deciding them."
print(f"Dataset check passed: {report1['n']} examples, labels {report1['label_counts']}.")

**Question:** how imbalanced is your data? If a model always predicted the most common label, what accuracy would it get? What does that tell you about using accuracy alone?

*Your answer:*

## Step 3: A fixed test set

Split into **train** (60%), **validation** (20%) and **test** (20%), keeping the label proportions equal in each. The **test set is sacred**: you never train on it, never tune on it, and never change it. To prove that it never changes, compute its **fingerprint** (a short code from its content). Every report you produce will carry the fingerprint, so anyone can see that two reports used the same test set. This cell is given.

In [ ]:
def stratified_split(examples, val_frac=0.2, test_frac=0.2, seed=0):
    rng = random.Random(seed); by_label = {}
    for e in examples: by_label.setdefault(e["label"], []).append(e)
    train, val, test = [], [], []
    for lab, items in by_label.items():
        rng.shuffle(items)
        n_test, n_val = round(len(items) * test_frac), round(len(items) * val_frac)
        test += items[:n_test]; val += items[n_test:n_test + n_val]; train += items[n_test + n_val:]
    for part in (train, val, test): rng.shuffle(part)
    return train, val, test

def fingerprint(test_set):
    """A short code that changes if even one test example changes."""
    return hashlib.sha256("\n".join(sorted(e["text"] + "|" + e["label"] for e in test_set)).encode()).hexdigest()[:12]

train_set, val_set, test_set = stratified_split(examples)
TEST_FINGERPRINT = fingerprint(test_set)
Path("test_fingerprint.txt").write_text(TEST_FINGERPRINT)
print(f"train {len(train_set)}, validation {len(val_set)}, test {len(test_set)}")
print("test set fingerprint:", TEST_FINGERPRINT)
for name, part in (("train", train_set), ("validation", val_set), ("test", test_set)):
    print(f"  {name:10s} {dict(Counter(e['label'] for e in part))}")

### Self-check: the split

In [ ]:
assert not ({e["text"] for e in train_set} & {e["text"] for e in test_set}), "Train and test share texts. The test set must be unseen."
assert not ({e["text"] for e in val_set} & {e["text"] for e in test_set}), "Validation and test share texts."
assert all(POSITIVE in {e["label"] for e in part} for part in (train_set, val_set, test_set)), "Every split must contain the rare label."
assert fingerprint(test_set) == TEST_FINGERPRINT, "The test set changed."
print("Split check passed. The test set is fixed.")

## Step 4: Metrics, from scratch

Libraries will compute metrics for you, but you only trust a number you understand. Write four small functions. Let the **positive** class be the rare label you care about.

- `accuracy(y_true, y_pred)`: the fraction of predictions that are right.
- `majority_baseline(y_true)`: the accuracy of a "model" that **always predicts the most common label**.
- `confusion_counts(y_true, y_pred, positive)`: returns `(tp, fp, fn, tn)`: true positives (predicted positive and really positive), false positives, false negatives, true negatives.
- `precision_recall_f1(tp, fp, fn)`: **precision** = tp / (tp + fp) ("when it says positive, how often is it right?"), **recall** = tp / (tp + fn) ("of all the real positives, how many did it find?"), **F1** = 2 x precision x recall / (precision + recall). If a denominator is zero, the value is 0.0.

In [ ]:
def accuracy(y_true, y_pred):
    # YOUR CODE HERE
    return 0.0

def majority_baseline(y_true):
    # YOUR CODE HERE
    return 0.0

def confusion_counts(y_true, y_pred, positive):
    """Return (tp, fp, fn, tn)."""
    # YOUR CODE HERE
    return 0, 0, 0, 0

def precision_recall_f1(tp, fp, fn):
    """Return (precision, recall, f1). Use 0.0 when a denominator is zero."""
    # YOUR CODE HERE
    return 0.0, 0.0, 0.0

### Self-check: metrics

In [ ]:
_t = [1, 1, 1, 0, 0, 0, 0, 0]; _p = [1, 1, 1, 1, 1, 0, 0, 0]
assert abs(accuracy(_t, _p) - 0.75) < 1e-9, f"6 of 8 are right, so accuracy is 0.75, not {accuracy(_t, _p)}."
assert abs(majority_baseline(["a"] * 9 + ["b"]) - 0.9) < 1e-9, "Always predicting the common label 'a' is right 9 times out of 10."
assert confusion_counts(_t, _p, 1) == (3, 2, 0, 3), f"Expected (tp, fp, fn, tn) = (3, 2, 0, 3) but got {confusion_counts(_t, _p, 1)}. A false positive is predicted positive but really negative; a false negative is the opposite."
_pr, _re, _f = precision_recall_f1(3, 2, 0)
assert abs(_pr - 0.6) < 1e-9, f"precision = tp / (tp + fp) = 3 / 5 = 0.6, but you got {_pr:.3f}."
assert abs(_re - 1.0) < 1e-9, f"recall = tp / (tp + fn) = 3 / 3 = 1.0, but you got {_re:.3f}."
assert abs(_f - 0.75) < 1e-9, f"F1 = 2 x precision x recall / (precision + recall) = 2 x 0.6 x 1.0 / 1.6 = 0.75, but you got {_f:.3f}."
assert precision_recall_f1(0, 0, 0) == (0.0, 0.0, 0.0), "Zero denominators must give 0.0, not an error."
assert precision_recall_f1(0, 3, 0) == (0.0, 0.0, 0.0), "With no true positives, precision, recall and F1 are all 0.0."
_t2 = ["normal"] * 9 + ["urgent"]; _lazy = ["normal"] * 10
assert accuracy(_t2, _lazy) == 0.9 and confusion_counts(_t2, _lazy, "urgent")[2] == 1 and precision_recall_f1(0, 0, 1)[1] == 0.0, "A model that never predicts the rare label has 90% accuracy and 0 recall."
print("metrics check passed. Note the last test: a model that never finds the rare class still gets 90% accuracy!")

## Step 5: The evaluation harness

Now combine everything into **one function** that anyone can run. Write `run_eval(predict_fn, test_set, positive, slice_key=None)`. `predict_fn(text)` returns the predicted label. The function returns a dictionary with:

- `n`, `accuracy`, `majority_baseline`,
- `tp`, `fp`, `fn`, `tn`, `precision`, `recall`, `f1` (positive class = `positive`),
- `slices`: the result of `slice_metrics(...)` (below) if `slice_key` is given, else `{}`,
- `failures`: a list of dictionaries `{"text", "label", "pred"}` for every wrong prediction,
- `test_fingerprint`: `fingerprint(test_set)`.

**First write `slice_metrics(examples, y_pred, key, positive)`**: it groups the examples by `example[key]` and returns a dictionary group -> `{"n", "precision", "recall"}` (the counts for the positive class, computed inside each group).

In [ ]:
def slice_metrics(examples, y_pred, key, positive):
    """Metrics per group: {group: {"n": ..., "precision": ..., "recall": ...}}."""
    # YOUR CODE HERE
    return {}

def run_eval(predict_fn, test_set, positive, slice_key=None):
    """Run the model on the test set and return the report dictionary described above."""
    # YOUR CODE HERE
    return {}

### Self-check: the harness

In [ ]:
_set = [{"text": "urgent help now", "label": "u", "channel": "chat"}, {"text": "all fine", "label": "n", "channel": "chat"},
        {"text": "server down", "label": "u", "channel": "chat"},   {"text": "thanks", "label": "n", "channel": "email"},
        {"text": "urgent: data lost", "label": "u", "channel": "email"}, {"text": "hello", "label": "n", "channel": "email"}]
_pred = lambda text: "u" if "urgent" in text else "n"          # finds the urgent ones that contain the word urgent
_sm = slice_metrics(_set, [_pred(e["text"]) for e in _set], "channel", "u")
assert set(_sm) == {"chat", "email"} and _sm["chat"]["n"] == 3 and _sm["email"]["n"] == 3, "slice_metrics must return one entry per group, with its size n."
assert abs(_sm["chat"]["recall"] - 0.5) < 1e-9 and abs(_sm["email"]["recall"] - 1.0) < 1e-9, f"Recall should be 0.5 in chat (1 of 2 urgent found) and 1.0 in email, got {_sm}."
assert abs(_sm["chat"]["precision"] - 1.0) < 1e-9, "Precision in chat: the one urgent prediction was right."
_rep = run_eval(_pred, _set, "u", slice_key="channel")
for _k in ("n", "accuracy", "majority_baseline", "tp", "fp", "fn", "tn", "precision", "recall", "f1", "slices", "failures", "test_fingerprint"):
    assert _k in _rep, f"The report is missing the key {_k!r}."
assert _rep["slices"] == _sm, "With slice_key given, the report must include slice_metrics(...) under the key slices."
assert (_rep["tp"], _rep["fp"], _rep["fn"], _rep["tn"]) == (2, 0, 1, 3), f"Expected tp, fp, fn, tn = 2, 0, 1, 3 but got {(_rep['tp'], _rep['fp'], _rep['fn'], _rep['tn'])}."
assert abs(_rep["accuracy"] - 5 / 6) < 1e-9 and abs(_rep["majority_baseline"] - 0.5) < 1e-9, "accuracy should be 5/6 and the majority baseline 0.5."
assert abs(_rep["recall"] - 2 / 3) < 1e-9 and abs(_rep["precision"] - 1.0) < 1e-9, "recall should be 2/3 and precision 1.0."
assert [f["text"] for f in _rep["failures"]] == ["server down"] and _rep["failures"][0]["pred"] == "n" and _rep["failures"][0]["label"] == "u", "failures must list each wrong prediction with text, label and pred."
assert _rep["test_fingerprint"] == fingerprint(_set), "Include the test set fingerprint."
assert run_eval(_pred, _set, "u")["slices"] == {}, "Without slice_key, slices must be an empty dictionary."
print("run_eval() check passed. You now own an evaluation harness.")

Two given helpers print a readable report and save it to a file. Then train the model under test (a simple TF-IDF and logistic regression classifier) and run **your harness on the validation set**.

In [ ]:
def print_report(r, title=""):
    print(f"=== {title} (n={r['n']}, data fingerprint {r['test_fingerprint']}) ===")
    print(f"accuracy {r['accuracy']:.3f}   (always-majority baseline {r['majority_baseline']:.3f})")
    print(f"precision {r['precision']:.3f}   recall {r['recall']:.3f}   F1 {r['f1']:.3f}   [tp {r['tp']}, fp {r['fp']}, fn {r['fn']}, tn {r['tn']}]")
    for g, m in sorted(r["slices"].items()):
        print(f"  slice {g:8s} n={m['n']:3d}  precision {m['precision']:.2f}  recall {m['recall']:.2f}")
    print(f"{len(r['failures'])} failures")

def save_report(r, path):
    Path(path).write_text(json.dumps({k: v for k, v in r.items() if k != "failures"}, indent=2))

def train_model(train_set, class_weight=None):
    """Train a simple text classifier and return a predict function: text -> label."""
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), LogisticRegression(max_iter=1000, class_weight=class_weight))
    pipe.fit([e["text"] for e in train_set], [e["label"] for e in train_set])
    return lambda text: pipe.predict([text])[0]

baseline_predict = train_model(train_set)
val_report = run_eval(baseline_predict, val_set, POSITIVE, slice_key=SLICE_KEY if SLICE_KEY in val_set[0] else None)
print_report(val_report, "baseline model on the VALIDATION set")

**Look at the numbers.** Compare the accuracy with the always-majority baseline, and then look at the recall of the rare class. Which number tells you how useful this model really is for the task? When would you care more about precision, and when more about recall? Write your answer:

*Your answer:*

## Step 6: Slices: who does the model fail?

An average hides unequal treatment. If the model works for one group of inputs and not for another, that is a **bias** problem, and it hurts real users. Look at the `slices` of the report above: the same metrics inside each group (here, the channel the message came from: email or chat).

**Questions:** does the model treat the groups equally? What could cause a gap (differences in writing style, in amount of training data, in label noise)? What would you do about it? And what other groups should *you* check for your own task (language, region, length, device, customer type)?

*Your answers:*

In [ ]:
print_report(val_report, "baseline on validation (look at the slices)")
if val_report["slices"]:
    gap = max(m["recall"] for m in val_report["slices"].values()) - min(m["recall"] for m in val_report["slices"].values())
    print(f"\nLargest recall gap between slices: {gap:.2f}")

## Step 7: Failure analysis

Metrics say *how much* the model fails. **Failure analysis** says *why*. Collect the failures (on the **validation** set, so that the test set stays untouched), tag each one with a **category**, count the categories, and fix the biggest one first.

Write `failure_breakdown(failures, rules)`. `rules` is a dictionary `name -> function(text) -> bool`. Each failure gets the name of the **first rule (in dictionary order) that matches its text**, or `"other"` if none match. Return a list of `(name, count)` pairs sorted by count from largest to smallest (ties alphabetically by name).

In [ ]:
def failure_breakdown(failures, rules):
    """Count failures per category. Return [(name, count), ...], largest first, ties sorted by name."""
    # YOUR CODE HERE
    return []

### Self-check: failure_breakdown

In [ ]:
_f = [{"text": "ok"}, {"text": "this is not urgent"}, {"text": "not now"}, {"text": "a long enough sentence here"}]
_rules = {"short": lambda t: len(t.split()) < 3, "negation": lambda t: " not " in " " + t.lower() + " "}
_b = failure_breakdown(_f, _rules)
assert _b == [("short", 2), ("negation", 1), ("other", 1)], f"Expected [('short', 2), ('negation', 1), ('other', 1)] but got {_b}. Use the FIRST matching rule, 'other' if none, and sort by count (largest first), then name."
assert failure_breakdown([], _rules) == [], "No failures should give an empty list."
print("failure_breakdown() check passed.")

Look at the actual failures and **write your own rules**. Print them first, read them, and notice patterns (very short messages? negation? a particular topic? a particular style?). Then define at least **three rules** as a dictionary named `rules` and run the breakdown. This cell prints the validation failures for you to read.

In [ ]:
failures = val_report["failures"]
print(f"{len(failures)} failures on the validation set. Read them:\n")
for f in failures[:25]:
    print(f"  true={f['label']:7s} predicted={f['pred']:7s} | {f['text']}")

In [ ]:
rules = {}     # YOUR CODE HERE: at least 3 rules, for example {"short chat message": lambda t: len(t.split()) < 7, ...}
breakdown = failure_breakdown(failures, rules)
print(breakdown)

### Self-check: your failure analysis

In [ ]:
assert len(failures) >= 10, (f"Only {len(failures)} validation failures. For this exercise you need at least 10 failures to analyze. "
                             "Use more data, or add the validation and test failures together after the final run.")
assert len(rules) >= 3 and all(callable(r) for r in rules.values()), "Define at least three rules, each a function that takes the text and returns True or False."
assert sum(c for _, c in breakdown) == len(failures), "Every failure must be counted exactly once."
_named = [c for n, c in breakdown if n != "other"]
assert _named and sum(_named) >= 0.5 * len(failures), (f"Your rules explain only {sum(_named)} of {len(failures)} failures, and the rest fall under 'other'. "
                                                          "Read the failures again and write rules that capture the patterns.")
print(f"Failure analysis check passed. Largest category: {breakdown[0][0]!r} with {breakdown[0][1]} of {len(failures)} failures.")

## Step 8: Propose a fix, then measure it fairly

**Propose one fix for the largest category** and write it down, then implement it. Common fixes: give the rare class more weight when training (`class_weight="balanced"`), collect more examples of the failing category, clean inconsistent labels, add a rule or a feature. Write your proposal in `PROPOSED_FIX` (at least two sentences: what you change, and why you expect it to help).

Then build `fix_predict`: a new predict function. If you use class weights, `train_model(train_set, class_weight="balanced")` already does it.

In [ ]:
PROPOSED_FIX = """
YOUR CODE HERE: describe the fix and why you expect it to help.
"""
fix_predict = baseline_predict     # YOUR CODE HERE: build the improved model

### Self-check: the fix

In [ ]:
assert "your code here" not in PROPOSED_FIX.lower() and len(PROPOSED_FIX.strip()) >= 100, "Describe your fix in at least two sentences: what you change and why you expect it to help."
assert callable(fix_predict) and fix_predict is not baseline_predict, "Build a new predict function for the fix (do not reuse baseline_predict)."
print("Fix check passed.")

## Step 9: The final report on the test set (given)

Now, **once**, run both models on the sacred **test set** with the *same* harness. Because the reports carry the fingerprint, anyone can verify that the comparison is fair.

In [ ]:
slice_arg = SLICE_KEY if SLICE_KEY in test_set[0] else None
baseline_report = run_eval(baseline_predict, test_set, POSITIVE, slice_arg)
fix_report = run_eval(fix_predict, test_set, POSITIVE, slice_arg)
print_report(baseline_report, "BASELINE on the TEST set")
print()
print_report(fix_report, "WITH THE FIX on the TEST set")
save_report(baseline_report, "report_baseline.json"); save_report(fix_report, "report_fix.json")

print("\nChange from baseline to fix:")
for k in ("accuracy", "precision", "recall", "f1"):
    print(f"  {k:9s} {baseline_report[k]:.3f} -> {fix_report[k]:.3f}  ({fix_report[k] - baseline_report[k]:+.3f})")

### Self-check: a fair comparison

In [ ]:
assert baseline_report["test_fingerprint"] == fix_report["test_fingerprint"] == TEST_FINGERPRINT == Path("test_fingerprint.txt").read_text(), "Both reports must use the same, unchanged test set."
assert baseline_report["n"] == fix_report["n"] == len(test_set), "Both reports must cover the whole test set."
print("Fair comparison check passed: both models were measured on the identical test set.")

**Write your conclusion.** Did the fix help? On which metric, and did it cost you something (for example precision)? Did it help the slices equally? Would you ship it? Be honest: a fix that does not help is a useful result.

*Your conclusion:*

## Stretch challenge: do two people agree?

If two people label the same items and often disagree, the labels are noisy and no model can be reliably better than the labels. **Cohen's kappa** measures agreement beyond chance. Write `cohens_kappa(a, b)` for two equally long lists of labels:

- observed agreement `po` = the fraction of positions where `a` and `b` are equal,
- chance agreement `pe` = the sum over labels of (fraction of `a` that is that label) x (fraction of `b` that is that label),
- `kappa = (po - pe) / (1 - pe)`. If `pe == 1`, return `1.0`.

Rough guide: above 0.8 is very good, 0.6 to 0.8 is good, below 0.4 means your guide is unclear.

In [ ]:
def cohens_kappa(a, b):
    """Agreement between two labelers, corrected for chance."""
    # YOUR CODE HERE
    return 0.0

### Self-check: kappa

In [ ]:
assert abs(cohens_kappa(["y", "y", "n", "n"], ["y", "n", "n", "n"]) - 0.5) < 1e-9, "po = 0.75, pe = 0.5 x 0.25 + 0.5 x 0.75 = 0.5, so kappa = (0.75 - 0.5) / (1 - 0.5) = 0.5."
assert cohens_kappa(["y", "n", "y"], ["y", "n", "y"]) == 1.0, "Perfect agreement gives kappa 1."
assert cohens_kappa(["y", "y"], ["y", "y"]) == 1.0, "If both labelers always give the same single label, return 1.0 (avoid dividing by zero)."
print("cohens_kappa() check passed.")

In [ ]:
if "label_b" in examples[0]:
    a = [e["label"] for e in examples]; b = [e["label_b"] for e in examples]
    print(f"Raw agreement {np.mean([x == y for x, y in zip(a, b)]):.1%}, Cohen's kappa {cohens_kappa(a, b):.2f}")
    dis = [e for e in examples if e["label"] != e["label_b"]]
    print(f"{len(dis)} disagreements. Read them, then ask: which rule is missing from the labeling guide?")
    for e in dis[:10]: print(f"  A={e['label']:7s} B={e['label_b']:7s} | {e['text']}")
else:
    print("Your data has no second labeler (label_b). Ask a friend to label 50 examples and compare.")

**Question:** what do the disagreements have in common? Which rule would you add to the labeling guide to remove them?

*Your answer:*

## Reflection and submission

Write three lines in your portfolio repo's `module-09/README.md`:
1. Your task and how many examples you labeled (and by whom)
2. Your baseline and fixed metrics on the test set, and the biggest failure category
3. One thing about your data that surprised you

Then download this notebook (*File > Download > .ipynb*), push it to `module-09` in your portfolio repo together with `report_baseline.json`, `report_fix.json` and your labeling guide (do **not** push private data), and submit the link in Moodle.